In [1]:
import panel as pn
pn.extension(comms='vscode')

import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Patch
from matplotlib.figure import Figure
import sys, os, warnings, copy
warnings.filterwarnings('ignore')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from secpi_main import (
    TwoLevelUrbanGrid,
    CorrectedCoolingModel,
    VariableRandomnessACO,
    TreeSpecies,
    SensitivityAnalyzer
)

print("Panel + SECPI loaded successfully.")
print(f"Panel version: {pn.__version__}")

Panel + SECPI loaded successfully.
Panel version: 1.8.7


In [2]:
# Global state container
class DashboardState:
    grid = None
    aco = None
    cooling_model = None
    history_best = None
    history_avg = None

state = DashboardState()


def make_grid_figure(grid):
    """Generate the CA grid visualization."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    color_map = {1: [0.6, 0.6, 0.6], 3: [0.6, 0.9, 0.6], 4: [0.95, 0.5, 0.5]}
    display_grid = np.ones((*grid.coarse_grid.shape, 3))
    for val, color in color_map.items():
        display_grid[grid.coarse_grid == val] = color

    ax1.imshow(display_grid, origin='lower', extent=[0, 100, 0, 100])
    ax1.set_title('CA-Generated Urban Grid', fontsize=12)
    ax1.set_xlabel('X (m)')
    ax1.set_ylabel('Y (m)')

    legend_patches = [
        Patch(facecolor=[0.6, 0.6, 0.6], label='Prohibited (Building)'),
        Patch(facecolor=[0.6, 0.9, 0.6], label='Available (Plantable)'),
        Patch(facecolor=[0.95, 0.5, 0.5], label='Vulnerable Zone'),
    ]
    ax1.legend(handles=legend_patches, loc='upper right', fontsize=8)

    unique, counts = np.unique(grid.coarse_grid, return_counts=True)
    stats = dict(zip(unique, counts))
    total = sum(counts)

    labels = ['Prohibited', 'Available', 'Vulnerable']
    values = [stats.get(1, 0), stats.get(3, 0), stats.get(4, 0)]
    bar_colors = ['gray', 'lightgreen', 'salmon']

    bars = ax2.bar(labels, values, color=bar_colors, edgecolor='black')
    for bar, val in zip(bars, values):
        pct = val / total * 100
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val}\n({pct:.0f}%)', ha='center', va='bottom',
                 fontweight='bold', fontsize=10)
    ax2.set_ylabel('Coarse Cells')
    ax2.set_title('Land Use Distribution')
    ax2.set_ylim(0, max(values) * 1.3 if max(values) > 0 else 10)

    plt.tight_layout()
    plt.close(fig)
    return fig


def make_solution_figure(grid, aco, cooling_model):
    """Generate the 4-panel optimization result."""
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

    tree_coords, tree_species = aco.best_solution
    display_cooling, _ = cooling_model.calculate_total_cooling(
        tree_coords, tree_species, grid.fine_grid_points,
        apply_competition=True
    )

    # Panel 1: Convergence
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(state.history_best, 'b-', linewidth=2, label='Best SECPI')
    ax1.plot(state.history_avg, 'r--', linewidth=1.5, alpha=0.7, label='Avg SECPI')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('SECPI')
    ax1.set_title('ACO Convergence')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Panel 2: Tree placements on grid
    ax2 = fig.add_subplot(gs[0, 1])
    for i in range(grid.coarse_height):
        for j in range(grid.coarse_width):
            x = j * grid.coarse_cell_size
            y = i * grid.coarse_cell_size
            lu = grid.coarse_grid[i, j]
            cmap = {1: 'gray', 3: 'lightgreen', 4: 'salmon'}
            rect = Rectangle(
                (x, y), grid.coarse_cell_size, grid.coarse_cell_size,
                facecolor=cmap.get(lu, 'white'),
                edgecolor='black', linewidth=0.3, alpha=0.6)
            ax2.add_patch(rect)

    ts = TreeSpecies()
    for (tx, ty), sp in zip(tree_coords, tree_species):
        color = ts.get_species_color(sp)
        ax2.scatter(tx, ty, color=color, s=120,
                    edgecolors='black', linewidth=1.5, zorder=5)
        cr = ts.get_crown_radius(sp)
        circle = Circle((tx, ty), cr, color=color, alpha=0.15, linewidth=1)
        ax2.add_patch(circle)

    ax2.set_xlim(0, grid.fine_width)
    ax2.set_ylim(0, grid.fine_height)
    ax2.set_aspect('equal')
    ax2.set_title('Tree Placements')

    # Panel 3: Cooling heatmap
    ax3 = fig.add_subplot(gs[1, 0])
    cg = display_cooling.reshape(grid.n_rows_fine, grid.n_cols_fine)
    im = ax3.imshow(cg.T, extent=[0, grid.fine_width, 0, grid.fine_height],
                    origin='lower', cmap='coolwarm_r', aspect='equal')
    for (tx, ty), sp in zip(tree_coords, tree_species):
        ax3.scatter(tx, ty, color='white', s=50,
                    edgecolors='black', linewidth=1, zorder=5)
    ax3.set_title('Cooling Distribution')
    plt.colorbar(im, ax=ax3, label='Cooling Intensity', shrink=0.8)

    # Panel 4: Results text
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('off')

    species_count = {}
    for sp in tree_species:
        species_count[sp] = species_count.get(sp, 0) + 1

    lines = [
        f"SECPI Score: {aco.best_secpi:.4f}",
        f"Trees Placed: {len(tree_coords)}",
        "",
        "Species Used:",
    ]
    for sp, count in species_count.items():
        cr = ts.get_crown_radius(sp)
        lines.append(f"  {sp} x{count} (r={cr:.1f}m)")
    lines.extend([
        "",
        "Cooling Statistics:",
        f"  Mean: {np.mean(display_cooling):.4f}",
        f"  Max:  {np.max(display_cooling):.4f}",
        f"  Std:  {np.std(display_cooling):.4f}",
    ])

    ax4.text(0.05, 0.95, "\n".join(lines),
             transform=ax4.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax4.set_title('Results Summary')

    plt.suptitle(f'SECPI = {aco.best_secpi:.4f}', fontsize=14, fontweight='bold')
    plt.close(fig)
    return fig


def make_equity_figure(grid, aco):
    """Generate zonal equity analysis."""
    cooling = aco.best_cooling
    scale = int(grid.coarse_cell_size / grid.fine_cell_size)
    fine_lu = np.repeat(np.repeat(grid.coarse_grid, scale, axis=0),
                        scale, axis=1).flatten()

    zones = {
        'Vulnerable': fine_lu == 4,
        'Available': fine_lu == 3,
        'Building': fine_lu == 1,
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    box_data = []
    box_labels = []
    box_colors = ['salmon', 'lightgreen', 'gray']
    for (name, mask), color in zip(zones.items(), box_colors):
        if np.any(mask):
            box_data.append(cooling[mask])
            box_labels.append(f"{name}\n(n={np.sum(mask):,})")

    bp = ax1.boxplot(box_data, labels=box_labels, patch_artist=True,
                     showfliers=False, widths=0.6)
    for patch, color in zip(bp['boxes'], box_colors[:len(box_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax1.set_ylabel('Cooling Intensity')
    ax1.set_title('Distribution by Zone')
    ax1.grid(axis='y', alpha=0.3)

    zone_means = {}
    for name, mask in zones.items():
        if np.any(mask):
            zone_means[name] = np.mean(cooling[mask])

    global_mean = np.mean(cooling)
    bars = ax2.bar(zone_means.keys(), zone_means.values(),
                   color=box_colors[:len(zone_means)], edgecolor='black', alpha=0.7)
    ax2.axhline(global_mean, color='navy', linestyle='--', linewidth=2,
                label=f'Global Mean ({global_mean:.4f})')
    for bar, val in zip(bars, zone_means.values()):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

    vuln_mean = zone_means.get('Vulnerable', 0)
    equity = vuln_mean / global_mean if global_mean > 0 else 0
    ax2.set_ylabel('Mean Cooling Intensity')
    ax2.set_title(f'Zonal Means (Equity Ratio: {equity:.2f})')
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.close(fig)
    return fig, equity


def make_decay_comparison_figure(decay_lambda):
    """All species radial decay comparison."""
    grid = state.grid
    if grid is None:
        return None

    cm = CorrectedCoolingModel(decay_lambda=decay_lambda)
    center = (grid.fine_width / 2, grid.fine_height / 2)
    ts = TreeSpecies()

    fig, ax = plt.subplots(figsize=(10, 5))
    for sp in ts.species_list:
        cooling = cm.calculate_cooling_contribution(
            center, sp, grid.fine_grid_points)
        distances = np.sqrt(
            (grid.fine_grid_points[:, 0] - center[0])**2 +
            (grid.fine_grid_points[:, 1] - center[1])**2
        )
        bins = np.linspace(0, 50, 80)
        bin_centers_list = []
        bin_vals = []
        for i in range(len(bins) - 1):
            mask = (distances >= bins[i]) & (distances < bins[i+1])
            if np.any(mask):
                bin_centers_list.append((bins[i] + bins[i+1]) / 2)
                bin_vals.append(np.mean(cooling[mask]))

        color = ts.SPECIES_DATA[sp]['color']
        cd = ts.SPECIES_DATA[sp]['crown_diameter_m']
        ax.plot(bin_centers_list, bin_vals, color=color, linewidth=2.5,
                label=f'{sp} (CD={cd:.0f}m)')

    ax.set_xlabel('Distance from Tree (m)', fontsize=11)
    ax.set_ylabel('Cooling Intensity', fontsize=11)
    ax.set_title(f'Species Radial Decay Comparison (λ={decay_lambda})', fontsize=13)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 50)
    plt.tight_layout()
    plt.close(fig)
    return fig

In [3]:
# ===== CA GRID CONTROLS =====
w_morphology = pn.widgets.Select(
    name='Morphology', options=['organic', 'linear'], value='organic')
w_p_init = pn.widgets.FloatSlider(
    name='Initial Density (p_init)', start=0.05, end=0.45, step=0.05, value=0.15)
w_ca_alpha = pn.widgets.FloatSlider(
    name='CA α (base prob)', start=0.0, end=0.5, step=0.05, value=0.1)
w_ca_beta = pn.widgets.FloatSlider(
    name='CA β (neighbor influence)', start=0.1, end=0.9, step=0.05, value=0.4)
w_seed = pn.widgets.IntSlider(
    name='Random Seed', start=1, end=999, step=1, value=42)

btn_generate = pn.widgets.Button(
    name='Generate Grid', button_type='primary', width=200)

# ===== COOLING MODEL CONTROLS =====
w_decay = pn.widgets.FloatSlider(
    name='λ (decay rate)', start=0.01, end=0.5, step=0.01, value=0.1)
w_cca_thresh = pn.widgets.FloatSlider(
    name='CCA Threshold (m²)', start=0.5, end=3.0, step=0.1, value=1.2)
w_comp_k = pn.widgets.FloatSlider(
    name='Competition K', start=1.0, end=15.0, step=0.5, value=5.0)
w_shade_w = pn.widgets.FloatSlider(
    name='Shade Weight', start=0.0, end=1.0, step=0.05, value=0.7)

# ===== ACO CONTROLS =====
w_n_trees = pn.widgets.IntSlider(
    name='Number of Trees', start=1, end=15, step=1, value=5)
w_n_species = pn.widgets.IntSlider(
    name='Species Pool Size', start=1, end=6, step=1, value=6)
w_n_ants = pn.widgets.IntSlider(
    name='Number of Ants', start=5, end=40, step=5, value=15)
w_n_iter = pn.widgets.IntSlider(
    name='Iterations', start=5, end=80, step=5, value=30)
w_q0 = pn.widgets.FloatSlider(
    name='q₀ (exploitation)', start=0.1, end=1.0, step=0.05, value=0.7)

btn_optimize = pn.widgets.Button(
    name='Run ACO Optimization', button_type='success', width=250)

# ===== OUTPUT PANES =====
grid_plot_pane = pn.pane.Matplotlib(plt.figure(), tight=True, dpi=100)
solution_plot_pane = pn.pane.Matplotlib(plt.figure(), tight=True, dpi=100)
equity_plot_pane = pn.pane.Matplotlib(plt.figure(), tight=True, dpi=100)
decay_plot_pane = pn.pane.Matplotlib(plt.figure(), tight=True, dpi=100)
status_pane = pn.pane.Alert('Ready. Generate a grid to begin.', alert_type='info')
results_pane = pn.pane.Str('No results yet.', styles={'font-family': 'monospace'})
plt.close('all')

In [4]:
def on_generate_grid(event):
    """Called when Generate Grid button is clicked."""
    status_pane.object = 'Generating grid...'
    status_pane.alert_type = 'warning'

    try:
        np.random.seed(w_seed.value)
        grid = TwoLevelUrbanGrid(
            coarse_width=10, coarse_height=10,
            coarse_cell_size=10.0, fine_cell_size=1.0
        )
        grid.generate_ca_archetype(
            params={
                'p_init': w_p_init.value,
                'alpha': w_ca_alpha.value,
                'beta': w_ca_beta.value,
                'theta': 3
            },
            morphology=w_morphology.value
        )
        state.grid = grid

        fig = make_grid_figure(grid)
        grid_plot_pane.object = fig

        # Also update the decay comparison
        decay_fig = make_decay_comparison_figure(w_decay.value)
        if decay_fig:
            decay_plot_pane.object = decay_fig

        unique, counts = np.unique(grid.coarse_grid, return_counts=True)
        stats = dict(zip(unique, counts))
        status_pane.object = (
            f'Grid generated. '
            f'P={stats.get(1,0)}, A={stats.get(3,0)}, V={stats.get(4,0)}. '
            f'{len(grid.plantable_coords)} plantable cells.'
        )
        status_pane.alert_type = 'success'

    except Exception as e:
        status_pane.object = f'Error generating grid: {str(e)}'
        status_pane.alert_type = 'danger'


def on_run_optimization(event):
    """Called when Run ACO button is clicked."""
    if state.grid is None:
        status_pane.object = 'Generate a grid first!'
        status_pane.alert_type = 'danger'
        return

    if len(state.grid.plantable_coords) == 0:
        status_pane.object = 'No plantable cells. Regenerate with lower p_init.'
        status_pane.alert_type = 'danger'
        return

    status_pane.object = (
        f'Running ACO: {w_n_ants.value} ants × {w_n_iter.value} iterations...'
    )
    status_pane.alert_type = 'warning'

    try:
        cooling_model = CorrectedCoolingModel(
            decay_lambda=w_decay.value,
            cca_threshold=w_cca_thresh.value,
            competition_k=w_comp_k.value,
            shade_weight=w_shade_w.value,
            evap_weight=1.0 - w_shade_w.value
        )
        state.cooling_model = cooling_model

        aco = VariableRandomnessACO(
            state.grid, cooling_model,
            n_trees=w_n_trees.value,
            n_ants=w_n_ants.value,
            n_iterations=w_n_iter.value,
            evaporation_rate=0.5,
            alpha=1.0, beta=2.0,
            q0=w_q0.value,
            random_seed=None,
            n_species_restricted=w_n_species.value
        )

        history_best, history_avg = aco.run(verbose=False)
        state.aco = aco
        state.history_best = history_best
        state.history_avg = history_avg

        if aco.best_solution:
            sol_fig = make_solution_figure(state.grid, aco, cooling_model)
            solution_plot_pane.object = sol_fig

            eq_fig, equity_ratio = make_equity_figure(state.grid, aco)
            equity_plot_pane.object = eq_fig

            tree_coords, tree_species = aco.best_solution
            display_cooling, _ = cooling_model.calculate_total_cooling(
                tree_coords, tree_species, state.grid.fine_grid_points,
                apply_competition=True
            )

            species_count = {}
            for sp in tree_species:
                species_count[sp] = species_count.get(sp, 0) + 1

            ts = TreeSpecies()
            result_lines = [
                f"SECPI Score:  {aco.best_secpi:.4f}",
                f"Equity Ratio: {equity_ratio:.2f}",
                f"Trees Placed: {len(tree_coords)}",
                "",
                "Species Breakdown:",
            ]
            for sp, count in species_count.items():
                cr = ts.get_crown_radius(sp)
                cpa = ts.SPECIES_DATA[sp]['CPA']
                result_lines.append(
                    f"  {sp} x{count}  (r={cr:.1f}m, CPA={cpa:.0f}m2)")
            result_lines.extend([
                "",
                "Cooling Stats:",
                f"  Mean:   {np.mean(display_cooling):.4f}",
                f"  Max:    {np.max(display_cooling):.4f}",
                f"  Std:    {np.std(display_cooling):.4f}",
                f"  Median: {np.median(display_cooling):.4f}",
                "",
                "Parameters Used:",
                f"  lambda={w_decay.value}, CCA_thresh={w_cca_thresh.value}",
                f"  comp_k={w_comp_k.value}, shade_w={w_shade_w.value}",
                f"  q0={w_q0.value}, ants={w_n_ants.value}, iter={w_n_iter.value}",
            ])
            results_pane.object = "\n".join(result_lines)

            status_pane.object = f'Optimization complete! SECPI = {aco.best_secpi:.4f}'
            status_pane.alert_type = 'success'
        else:
            status_pane.object = 'ACO found no valid solution.'
            status_pane.alert_type = 'danger'

    except Exception as e:
        status_pane.object = f'Error during optimization: {str(e)}'
        status_pane.alert_type = 'danger'


def on_decay_change(event):
    """Update decay comparison when lambda slider changes."""
    if state.grid is not None:
        fig = make_decay_comparison_figure(event.new)
        if fig:
            decay_plot_pane.object = fig


# Wire buttons
btn_generate.on_click(on_generate_grid)
btn_optimize.on_click(on_run_optimization)
w_decay.param.watch(on_decay_change, 'value')

print("Callbacks wired. Ready to build layout.")

Callbacks wired. Ready to build layout.


In [5]:
# ===== SIDEBAR =====
sidebar = pn.Column(
    pn.pane.Markdown("## Controls"),

    pn.pane.Markdown("### Urban Grid (CA)"),
    w_morphology, w_p_init, w_ca_alpha, w_ca_beta, w_seed,
    btn_generate,

    pn.layout.Divider(),

    pn.pane.Markdown("### Cooling Model"),
    w_decay, w_cca_thresh, w_comp_k, w_shade_w,

    pn.layout.Divider(),

    pn.pane.Markdown("### ACO Optimizer"),
    w_n_trees, w_n_species, w_n_ants, w_n_iter, w_q0,
    btn_optimize,

    width=320,
    margin=(10, 15)
)

# ===== MAIN CONTENT WITH TABS =====
tabs = pn.Tabs(
    ('Grid', pn.Column(
        pn.pane.Markdown("### CA-Generated Urban Layout"),
        grid_plot_pane
    )),
    ('Species Decay', pn.Column(
        pn.pane.Markdown("### Radial Cooling Decay by Species"),
        pn.pane.Markdown("*Adjust λ slider on the left to update.*"),
        decay_plot_pane
    )),
    ('Optimization', pn.Column(
        pn.pane.Markdown("### ACO Results"),
        solution_plot_pane
    )),
    ('Equity Analysis', pn.Column(
        pn.pane.Markdown("### Zonal Cooling Equity"),
        equity_plot_pane
    )),
    ('Results Data', pn.Column(
        pn.pane.Markdown("### Numerical Results"),
        results_pane
    )),
    dynamic=True
)

# ===== FULL DASHBOARD =====
dashboard = pn.Row(
    sidebar,
    pn.Column(
        pn.pane.Markdown("# SECPI Interactive Dashboard"),
        pn.pane.Markdown(
            "*Synergistic & Equitable Cooling Performance Index — "
            "Tree Placement Optimizer*"
        ),
        status_pane,
        tabs,
        margin=(10, 15)
    )
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'360a1e1d-0f1b-4828-8838-3ad69c74c3c6': {'version…

---
### How to Use
1. Adjust the **Urban Grid** sliders, click **Generate Grid**
2. Switch to the **Species Decay** tab to see how cooling drops with distance
3. Adjust **Cooling Model** and **ACO** parameters
4. Click **Run ACO Optimization**
5. Check **Optimization**, **Equity Analysis**, and **Results Data** tabs

Changes to `secpi_main.py` are picked up after a kernel restart (circular arrow icon at top).